# Plotting Comparisons between U-FNO and AU-Net
## (Figures 4-12 in Morphew et al. 2026)

# User settings, and HDF5 File Loading

This next code block will handle creating the test DataLoader for inference. This requires a few steps.

1. This code block will filter and calculate normalization stats for the HDF5 file.
2. It will then create a custom dataset object containing both the HDF5 inputs and outputs.
3. It will then split the train, validation, and train datasets (The seed is set to 0 so this won't vary between runs).

In [ ]:
import os
import numpy as np
from math import exp
from datetime import datetime
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter


from data_utils import prepare_dataloaders
from U_FNO import UFNO3d
from AU_NET import UNet3DAttention

# User Settings
BATCH_SIZE = 2
NORMALIZATION_METHOD = 'z-score'  # 'z-score' or 'min-max'
INPUT_HDF5 = 'small_input_arrays.hdf5'
OUTPUT_HDF5 = 'small_output_arrays.hdf5'
STATS_PATH = 'normalization_stats'
CHECKPOINT_PATH = 'checkpoints_anuet'
FIGURE_FOLDER = 'plots_test_dataset'
# The pickle file paths for saving stats
INPUT_Z_SCORE_STATS_FILE = os.path.join(STATS_PATH, 'input_stats_z_score.pkl')
OUTPUT_Z_SCORE_STATS_FILE = os.path.join(STATS_PATH, 'output_stats_z_score.pkl')
INPUT_MIN_MAX_STATS_FILE = os.path.join(STATS_PATH, 'input_stats_min_max.pkl')
OUTPUT_MIN_MAX_STATS_FILE = os.path.join(STATS_PATH, 'output_stats_min_max.pkl')

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

training_loader, validation_loader, test_loader = prepare_dataloaders(
    input_hdf5=INPUT_HDF5,
    output_hdf5=OUTPUT_HDF5,
    batch_size=BATCH_SIZE,
    norm_method=NORMALIZATION_METHOD,
    stats_path=STATS_PATH
)

print(f"DataLoaders are ready. Train size: {len(training_loader.dataset)}")

# Loading Trained Models

In [ ]:
from U_FNO import UFNO3d
from AU_NET import UNet3DAttention

# model hyperparameters
mode1 = 16
mode2 = 12
mode3 = 4
width = 8
in_channels = 6
out_channels = 2

# create model

model_aunet = UNet3DAttention(
            in_channels=in_channels, 
            out_channels=out_channels,
            base_c=width
        )

UNet = True
if UNet: 
    model_ufno = UFNO3d(mode1, mode2, mode3, width, in_channels=in_channels, out_channels=out_channels, UNet = True)
else:
        model_ufno = UFNO3d(mode1, mode2, mode3, width, in_channels=in_channels, out_channels=out_channels, UNet = False)


# Step 2: Load the saved state dictionary
model_ufno.load_state_dict(torch.load('checkpoints_ufno/model_best_snapshot_46000.pt', weights_only=True))
model_aunet.load_state_dict(torch.load('checkpoints_aunet/model_best_snapshot.pt', weights_only=True))
model_aunet.to(device)
model_ufno.to(device)

# Running inference on the test data (U-FNO)

In [ ]:
# Helper function to unscale outputs
def unscale_output(tensor, stats_file_path, type='z-score'):
    """
    Applies the inverse z-score transformation or min-max transformation using saved stats.
    """
    with open(stats_file_path, 'rb') as f:
        stats = pickle.load(f)
        if type == 'z-score':
            mean = stats['mean'].to(tensor.device)
            std = stats['std'].to(tensor.device)
        elif type == 'min-max':
            min = stats['min'].to(tensor.device)
            max = stats['max'].to(tensor.device)
    epsilon = 1e-8
    if type == 'z-score':
        return tensor * (std + epsilon) + mean
    elif type == 'min-max':
        return tensor * (max - min + epsilon) + min 


model_ufno.to(device)
model_ufno.eval()

all_predictions_unscaled = []
all_labels_unscaled = []

with torch.no_grad():
    for i, data_pair in enumerate(test_loader):
        inputs_data, labels_data = data_pair

        # Move data to the correct device
        inputs = inputs_data.squeeze(0).to(device)
        labels_scaled = labels_data.squeeze(0).to(device)


        # Forward pass to get model outputs in scaled space
        outputs_scaled = model_ufno(inputs)
        # Unscale both the predictions and the labels
        if NORMALIZATION_METHOD == 'z-score':
            outputs_unscaled = unscale_output(outputs_scaled, OUTPUT_Z_SCORE_STATS_FILE)
            labels_unscaled = unscale_output(labels_scaled, OUTPUT_Z_SCORE_STATS_FILE)

        elif NORMALIZATION_METHOD == 'min-max':
            outputs_unscaled = unscale_output(outputs_scaled, OUTPUT_MIN_MAX_STATS_FILE, 'min-max')
            labels_unscaled = unscale_output(labels_scaled, OUTPUT_MIN_MAX_STATS_FILE, 'min-max')
        else:
            outputs_unscaled = outputs_scaled
            labels_unscaled = labels_scaled
        # Move unscaled results to CPU and convert to numpy arrays
        all_predictions_unscaled.append(outputs_unscaled.cpu().numpy())
        all_labels_unscaled.append(labels_unscaled.cpu().numpy())

predictions_np_ufno = np.concatenate(all_predictions_unscaled, axis=0)
labels_np_ufno = np.concatenate(all_labels_unscaled, axis=0)

print("\nInference and unscaling complete (U-FNO).")
print(f"Shape of all predictions: {predictions_np_ufno.shape}")
print(f"Shape of all labels: {labels_np_ufno.shape}")

# Running inference on the test data (AU-Net)

In [ ]:
# Helper function to unscale outputs
def unscale_output(tensor, stats_file_path, type='z-score'):
    """
    Applies the inverse z-score transformation using saved stats.
    """
    with open(stats_file_path, 'rb') as f:
        stats = pickle.load(f)
        if type == 'z-score':
            mean = stats['mean'].to(tensor.device)
            std = stats['std'].to(tensor.device)
        elif type == 'min-max':
            min = stats['min'].to(tensor.device)
            max = stats['max'].to(tensor.device)
    epsilon = 1e-8
    if type == 'z-score':
        return tensor * (std + epsilon) + mean
    elif type == 'min-max':
        return tensor * (max - min + epsilon) + min 

# Find the best model saved during training
best_model_path = None
for f in os.listdir(CHECKPOINT_PATH):
    if f.startswith('model_best_'):
        best_model_path = os.path.join(CHECKPOINT_PATH, f)
        break

model_aunet.to(device)
model_aunet.eval()

all_predictions_unscaled = []
all_labels_unscaled = []

with torch.no_grad():
    for i, data_pair in enumerate(test_loader):
        inputs_data, labels_data = data_pair
        # Move data to the correct device
        inputs = inputs_data.squeeze(0).to(device)
        labels_scaled = labels_data.squeeze(0).to(device)
        #print(np.max(labels_scaled))

        # Forward pass to get model outputs in scaled space
        outputs_scaled = model_aunet(inputs)
        # UNSCALE both the predictions and the labels
        if NORMALIZATION_METHOD == 'z-score':
            outputs_unscaled = unscale_output(outputs_scaled, OUTPUT_Z_SCORE_STATS_FILE)
            labels_unscaled = unscale_output(labels_scaled, OUTPUT_Z_SCORE_STATS_FILE)

        elif NORMALIZATION_METHOD == 'min-max':
            outputs_unscaled = unscale_output(outputs_scaled, OUTPUT_MIN_MAX_STATS_FILE, 'min-max')
            labels_unscaled = unscale_output(labels_scaled, OUTPUT_MIN_MAX_STATS_FILE, 'min-max')
        else:
            outputs_unscaled = outputs_scaled
            labels_unscaled = labels_scaled
        # Move unscaled results to CPU and convert to numpy arrays
        all_predictions_unscaled.append(outputs_unscaled.cpu().numpy())
        all_labels_unscaled.append(labels_unscaled.cpu().numpy())

predictions_np_aunet = np.concatenate(all_predictions_unscaled, axis=0)
labels_np_aunet = np.concatenate(all_labels_unscaled, axis=0)

print("\nInference and unscaling complete (AU-Net).")
print(f"Shape of all predictions: {predictions_np_aunet.shape}")
print(f"Shape of all labels: {labels_np_aunet.shape}")

# 1:1 Comparison Plot: Spatial Average of Concentration Prediction per time sample

In [ ]:
channel_to_plot = 1


summed_preds_ufno = np.mean(predictions_np_ufno[:, :, :, :, channel_to_plot], axis=(1, 2))
summed_actuals_ufno = np.mean(labels_np_ufno[:, :, :, :, channel_to_plot], axis=(1, 2))

summed_preds_aunet = np.mean(predictions_np_aunet[:, :, :, :, channel_to_plot], axis=(1, 2))
summed_actuals_aunet = np.mean(labels_np_aunet[:, :, :, :, channel_to_plot], axis=(1, 2))

num_timesteps = predictions_np_ufno.shape[3]
num_simulations = predictions_np_ufno.shape[0]

y_data_ufno = summed_preds_ufno.flatten()
x_data_ufno = summed_actuals_ufno.flatten()

y_data_aunet = summed_preds_aunet.flatten()
x_data_aunet = summed_actuals_aunet.flatten()

# MAE calculation
mae_aunet = np.mean(np.abs(y_data_aunet - x_data_aunet))
mae_ufno = np.mean(np.abs(y_data_ufno - x_data_ufno))

# Plot color setup
timesteps = np.arange(num_timesteps)
colors_data = np.tile(timesteps, num_simulations)

plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(1, 2, figsize=(13.33, 7.5))

scatter0 = ax[0].scatter(x_data_aunet, y_data_aunet, c=colors_data, cmap='viridis', alpha=0.2, s=15)
scatter1 = ax[1].scatter(x_data_ufno, y_data_ufno, c=colors_data, cmap='viridis', alpha=0.2, s=15)

cbar_ax = fig.add_axes([0.92, 0.1, 0.03, 0.8]) 
cbar = fig.colorbar(scatter1, cax=cbar_ax)
cbar.set_label('Timestep', fontsize=12, weight='bold')

# Annotations
ax[0].set_xlabel('Average of Actual Concentration Values ($\\mu g/L$)', fontsize=12)
ax[0].set_ylabel('Average of Predicted Concentration Values ($\\mu g/L$)', fontsize=12)
ax[0].text(0.05, 0.95, f'MAE AUNET ($\\mu g/L$): {mae_aunet:.2f}', 
           transform=ax[0].transAxes, fontsize=12, verticalalignment='top', 
           bbox=dict(boxstyle='round,pad=0.5', fc='wheat', alpha=0.5))


ax[1].set_xlabel('Average of Actual Concentration Values ($\\mu g/L$)', fontsize=12)
ax[1].text(0.05, 0.95, f'MAE UFNO ($\\mu g/L$): {mae_ufno:.2f}', 
           transform=ax[1].transAxes, fontsize=12, verticalalignment='top', 
           bbox=dict(boxstyle='round,pad=0.5', fc='wheat', alpha=0.5))

# 1:1 line
for a in ax:
    min_val = min(a.get_xlim()[0], a.get_ylim()[0])
    max_val = max(a.get_xlim()[1], a.get_ylim()[1])
    a.plot([min_val, max_val], [min_val, max_val], 'r--', label='1:1 Line')
    a.legend(loc='lower right')

plt.savefig('{}/all_simulations_by_timestep.png'.format(FIGURE_FOLDER), dpi=300)


# 1:1 Plots for all pixels in all test samples

In [ ]:
num_channels = predictions_np_ufno.shape[4]

num_timesteps = predictions_np_ufno.shape[3]
num_y = predictions_np_ufno.shape[1]
num_x = predictions_np_ufno.shape[2]
num_simulations = predictions_np_ufno.shape[0]
print("--- Generating Scatter Plots Colored by Timestep (This will take a while) ---")
units = ['(m)', '($\\mu g/L$)']
channel_name = ['Head', 'Concentration']
for channel_idx in range(num_channels):
   # Select and Flatten Data
   preds_for_channel_ufno = predictions_np_ufno[..., channel_idx]
   actuals_for_channel_ufno = labels_np_ufno[..., channel_idx]

   y_data_ufno = preds_for_channel_ufno.flatten()
   x_data_ufno = actuals_for_channel_ufno.flatten()
   
   mae_ufno = np.mean(np.abs(y_data_ufno - x_data_ufno))

   preds_for_channel_aunet = predictions_np_aunet[..., channel_idx]
   actuals_for_channel_aunet = labels_np_aunet[..., channel_idx]

   y_data_aunet = preds_for_channel_aunet.flatten()
   x_data_aunet = actuals_for_channel_aunet.flatten()
   
   mae_aunet = np.mean(np.abs(y_data_aunet - x_data_aunet))

   # Create Timestep Colors
   # Create a 1D tensor representing the timesteps [0, 1, ..., 99]
   ts = torch.arange(num_timesteps)
   
   # Reshape the timesteps tensor so it can be broadcast across the other dimensions (sims, y, x)
   # New shape: (1, 100, 1, 1)
   ts_reshaped = ts.view(1, num_timesteps, 1, 1)
   # Expand the tensor to match the shape of the data for one channel, without copying data
   # New shape: (3, 100, 20, 20)
   timesteps_tensor = ts_reshaped.expand(num_simulations, num_timesteps, num_y, num_x)
   # Flatten the timesteps tensor to create the color array. This ensures each data point
   # is mapped to its correct timestep value.
   colors_data = timesteps_tensor.flatten().numpy()
   num_total_points = len(x_data_ufno)
   # Choose a manageable number of points to plot (for many points, this plot becomes impossible to render)
   num_sample_points = min(num_total_points, 50000) 

   # Generate random indices
   indices = np.random.choice(num_total_points, num_sample_points, replace=False)

   # Sample all data arrays using these indices
   x_data_ufno = x_data_ufno[indices]
   y_data_ufno = y_data_ufno[indices]

   x_data_aunet = x_data_aunet[indices]
   y_data_aunet = y_data_aunet[indices]
   colors_data = colors_data[indices]
   plt.style.use('seaborn-v0_8-whitegrid')
   fig, ax = plt.subplots(1, 2, figsize=(13.33, 7.5))
   
   scatter = ax[0].scatter(x_data_aunet, y_data_aunet, c=colors_data, cmap='viridis', alpha=0.4, s=5, rasterized=True)
   scatter = ax[1].scatter(x_data_ufno, y_data_ufno, c=colors_data, cmap='viridis', alpha=0.4, s=5, rasterized=True)
   
   # Add a colorbar to show the mapping of color to timestep
   cbar_ax = fig.add_axes([0.92, 0.1, 0.03, 0.8]) 
   cbar = fig.colorbar(scatter, cax=cbar_ax)
   cbar.set_label('Timestep', fontsize=12, weight='bold')
   
   # Ref line and labels
   ax[0].set_xlabel(f'Actual {channel_name[channel_idx]} Values: {units[channel_idx]}', fontsize=12)
   ax[0].set_ylabel(f'Predicted {channel_name[channel_idx]} Values: {units[channel_idx]}', fontsize=12)
   ax[0].text(0.05, 0.95, f'MAE AUNET ($\\mu g/L$): {mae_aunet:.2f}', 
           transform=ax[0].transAxes, fontsize=12, verticalalignment='top', 
           bbox=dict(boxstyle='round,pad=0.5', fc='wheat', alpha=0.5))

   ax[1].set_xlabel(f'Actual {channel_name[channel_idx]} Values: {units[channel_idx]}', fontsize=12)
   ax[1].text(0.05, 0.95, f'MAE UFNO ($\\mu g/L$): {mae_ufno:.2f}', 
           transform=ax[1].transAxes, fontsize=12, verticalalignment='top', 
           bbox=dict(boxstyle='round,pad=0.5', fc='wheat', alpha=0.5))

   min_val = min(ax[0].get_xlim()[0], ax[0].get_ylim()[0])
   max_val = max(ax[0].get_xlim()[1], ax[0].get_ylim()[1])
   ax[0].plot([min_val, max_val], [min_val, max_val], 'r--', label='1:1 Line')
   ax[0].legend(loc='lower right')
   if channel_idx == 0:
      ax[0].set_xlim(min_val, max_val)
      ax[0].set_ylim(min_val, max_val)
   else:
      ax[0].set_xlim(0, max_val)
      ax[0].set_ylim(0, max_val)

   min_val = min(ax[1].get_xlim()[0], ax[1].get_ylim()[0])
   max_val = max(ax[1].get_xlim()[1], ax[1].get_ylim()[1])
   ax[1].plot([min_val, max_val], [min_val, max_val], 'r--', label='1:1 Line')
   ax[1].legend(loc='lower right')
   if channel_idx == 0:
      ax[1].set_xlim(min_val, max_val)
      ax[1].set_ylim(min_val, max_val)
   else:
      ax[1].set_xlim(0, max_val)
      ax[1].set_ylim(0, max_val)
   
   # Save the figure
   plt.savefig(f'{FIGURE_FOLDER}/channel_{channel_idx + 1}_scatter_by_timestep.png', dpi=300)


print("Scatter plots saved as 'channel_..._scatter_by_timestep.png'")


# 1:1 Comparison Plot: Well Locations

This loads a CSV containing locations of certain wells

In [ ]:
# plot by wells

# the .csv file contains the z, y, x values for each well. 
# Path to the .csv file
csv_file = "crm_new_gwf_obs_clean.csv"
num_channels = predictions_np_ufno.shape[4]

In [ ]:
wells_df = pd.read_csv(csv_file)
well_names = wells_df.columns

print(f"Processing {len(well_names)} wells for {num_channels} channels...")

for channel_idx in range(num_channels):
    print(f"\n--- Channel {channel_idx + 1} ---")

    for i in range(len(well_names)):
        try:
            full_well_name = well_names[i]
            parts = full_well_name.split('_')
            
            # Grab x and y from the right using negative indices
            x_grid = int(parts[-1]) # The last element is 'x'
            y_grid = int(parts[-2]) # The second-to-last element is 'y'

            clean_well_name = parts[0]

            y_idx = y_grid - 1
            x_idx = x_grid - 1

            print(f"Plotting well: {clean_well_name} at (y={y_idx}, x={x_idx})")

            preds_at_well_aunet = predictions_np_aunet[:, y_idx, x_idx, :, channel_idx]
            actuals_at_well_aunet = labels_np_aunet[:, y_idx, x_idx, :, channel_idx]

            preds_at_well_ufno = predictions_np_ufno[:, y_idx, x_idx, :, channel_idx]
            actuals_at_well_ufno = labels_np_ufno[:, y_idx, x_idx, :, channel_idx]

            y_data_aunet = preds_at_well_aunet.flatten()
            x_data_aunet = actuals_at_well_aunet.flatten()

            y_data_ufno = preds_at_well_ufno.flatten()
            x_data_ufno = actuals_at_well_ufno.flatten()

            rmse_aunet = np.sqrt(np.mean((x_data_aunet-y_data_aunet)**2))
            rmse_ufno = np.sqrt(np.mean((x_data_ufno-y_data_ufno)**2))

            plt.style.use('seaborn-v0_8-whitegrid')
            fig, ax = plt.subplots(1, 2, figsize=(13.33, 7.5))

            ax[0].scatter(x_data_aunet, y_data_aunet, alpha=0.4, s=20)
            ax[1].scatter(x_data_ufno, y_data_ufno, alpha=0.4, s=20)
            if channel_idx == 0:
                ax[0].set_xlabel('Actual Head Values at Well (m)', fontsize=12)
                ax[0].set_ylabel('Predicted Head Values at Well (m)', fontsize=12)
                #ax[0].set_title(f'AUNET Head Predictions against - Well: {i}', fontsize=14, weight='bold')

                ax[1].set_xlabel('Actual Head Values at Well (m)', fontsize=12)
                #ax[1].set_title(f'UFNO Channel {channel_idx + 1} - Well: {i}', fontsize=14, weight='bold')
            if channel_idx == 1:
                ax[0].set_xlabel('Actual Concentration Values at Well ($\\mu g/L$)', fontsize=12)
                ax[0].set_ylabel('Predicted Concentration Values at Well ($\\mu g/L$)', fontsize=12)
                #ax[0].set_title(f'AUNET Head Predictions against - Well: {i}', fontsize=14, weight='bold')

                ax[1].set_xlabel('Actual Concentration Values at Well ($\\mu g/L$)', fontsize=12)
                #ax[1].set_title(f'UFNO Channel {channel_idx + 1} - Well: {i}', fontsize=14, weight='bold')
            
            
            ax[0].text(0.05, 0.95, f'RMSE AUNET: {rmse_aunet:.2f}', transform=ax[0].transAxes, fontsize=12,
                    verticalalignment='top', bbox=dict(boxstyle='round,pad=0.5', fc='wheat', alpha=0.5))

            ax[1].text(0.05, 0.95, f'RMSE UFNO: {rmse_ufno:.2f}', transform=ax[1].transAxes, fontsize=12,
                    verticalalignment='top', bbox=dict(boxstyle='round,pad=0.5', fc='wheat', alpha=0.5))
            
            min_val = min(x_data_aunet.min(), y_data_aunet.min())
            max_val = max(x_data_aunet.max(), y_data_aunet.max())
            ax[0].plot([min_val, max_val], [min_val, max_val], 'r--', label='1:1 Line')
            ax[0].legend(loc='lower right')
            ax[0].axis('equal')

            min_val = min(x_data_ufno.min(), y_data_ufno.min())
            max_val = max(x_data_ufno.max(), y_data_ufno.max())
            ax[1].plot([min_val, max_val], [min_val, max_val], 'r--', label='1:1 Line')
            ax[1].legend(loc='lower right')
            ax[1].axis('equal')

            plot_filename = f'{FIGURE_FOLDER}/wells/csv_channel_{channel_idx + 1}_well_{clean_well_name}.png'
            plt.savefig(plot_filename, dpi=300)
            plt.close(fig)

        except (IndexError, ValueError) as e:
            print(f"Could not parse well name '{full_well_name}'. Skipping. Error: {e}")

print(f"\n✅ All plots saved successfully.")

In [ ]:
def plot_channel_grid(data, sample_idx, channel_idx, ground_truth, model_type, vmin=None, vmax=None):
    """
    Plots a grid of images with a thinned, horizontal colorbar placed in the 
    center of the unused bottom-right panel of the 3x4 grid using inset_axes.
    """
    # Extract the specific sample
    sample_data = data[sample_idx]
    num_timesteps = sample_data.shape[2] 

    # Grid Setup (3x4 for 12 total spaces)
    rows = 3
    cols = 4
    
    # Use constrained_layout=True for automatic spacing
    fig, axes = plt.subplots(rows, cols, figsize=(13.33, 7.5), constrained_layout=True)
    
    axes_flat = axes.flatten()

    # Find the min and max values (handle potential NaNs)
    if vmin is None:
        vmin = np.nanmin(sample_data[:, :, :, channel_idx])
    if vmax is None:
        vmax = np.nanmax(sample_data[:, :, :, channel_idx])

    im = None 
    for t in range(num_timesteps):
        image_slice = sample_data[:, :, t, channel_idx]
        ax = axes_flat[t]
        im = ax.imshow(image_slice, cmap='viridis', vmin=vmin, vmax=vmax)
        ax.set_title(f"Timestep {t+1}")
        ax.axis('off') 

    # cbar placement    
    parent_cbar_ax = axes_flat[rows * cols - 1] 
    parent_cbar_ax.axis('off') 

    for i in range(num_timesteps, len(axes_flat) - 1):
        axes_flat[i].axis('off')
    
    cbar_ax = inset_axes(
        parent_cbar_ax, 
        width="90%",    
        height="10%",  
        loc='center'     
    )

    cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal') 
    
    if channel_idx == 0:
        cbar.set_label('Head (m)')
        channel_label = 'Head'
    else:
        cbar.set_label('Concentration ($\\mu g/L$)')
        channel_label = 'Concentration'
        
    # saving logic
    if model_type == 'aunet':
        folder = 'samples_aunet'
    else: 
        folder =  'samples_ufno'
    if ground_truth:
        file_suffix = 'ground_truth'
    else:
        file_suffix = 'model_prediction'
    print("saving figure")
    plt.savefig(f'{FIGURE_FOLDER}/{folder}/{channel_label}_{sample_idx+1}_{file_suffix}.png', dpi=300)
    
    plt.close()

In [ ]:
for i in range(len(labels_np_aunet)):
    print("Generating images for AU-NET sample #", i)
    sample_to_plot = i

    # Plot predictions and ground truth for the FIRST channel (index 0)
    plot_channel_grid(predictions_np_aunet, sample_to_plot, 0, False, 'aunet')
    plot_channel_grid(labels_np_aunet, sample_to_plot, 0, True, 'aunet')

    # Plot predictions and ground truth for the SECOND channel (index 1)
    plot_channel_grid(predictions_np_aunet, sample_to_plot, 1, False, 'aunet', vmin=0, vmax=1400)
    plot_channel_grid(labels_np_aunet, sample_to_plot, 1, True, 'aunet', vmin=0, vmax=1400)


for i in range(len(labels_np_ufno)):
    print("Generating images for U-FNO sample #", i)
    sample_to_plot = i

    # Plot predictions and ground truth for the FIRST channel (index 0)
    plot_channel_grid(predictions_np_ufno, sample_to_plot, 0, False, 'ufno')
    plot_channel_grid(labels_np_ufno, sample_to_plot, 0, True, 'ufno')

    # Plot predictions and ground truth for the SECOND channel (index 1)
    plot_channel_grid(predictions_np_ufno, sample_to_plot, 1, False, 'ufno', vmin=0, vmax=1400)
    plot_channel_grid(labels_np_ufno, sample_to_plot, 1, True, 'ufno', vmin=0, vmax=1400)